In [ ]:
import pandas as pd

df = pd.read_csv('./files/results/data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48447 entries, 0 to 48446
Data columns (total 75 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Unnamed: 0                            48447 non-null  int64  
 1   id_discente                           48447 non-null  int64  
 2   ano                                   48447 non-null  int64  
 3   periodo                               48447 non-null  int64  
 4   id_disciplina                         48447 non-null  int64  
 5   situacao                              48447 non-null  str    
 6   sexo                                  19588 non-null  str    
 7   estado_civil                          19588 non-null  str    
 8   raca_declarada                        19588 non-null  str    
 9   discente_nivel                        19588 non-null  str    
 10  id_curso                              19588 non-null  float64
 11  id_curriculo              

In [93]:
df_filter = df.copy()

# TRATANDO OS DADOS
df_filter['ano_ingresso'] = df_filter['ano_ingresso'].fillna(df['ano_primeiro_registro'])
df_filter['periodo_ingresso'] = df_filter['periodo_ingresso'].fillna(df['periodo_primeiro'])
df_filter['forma_ingresso'] = df_filter['forma_ingresso'].fillna("PROCESSO SELETIVO")
df_filter['ativo'] = df_filter['ativo'].fillna(True)
df_filter['excluir_avaliacao_institucional'] = df_filter['excluir_avaliacao_institucional'].fillna(False)


df_filter = df_filter.drop([
    'Unnamed: 0', 'id_curriculo', 'curso_grande_area_conhecimento', 'codigo_emec', 'discente_nivel',
    "id_discente","id_disciplina","id_detalhe",
    "codigo","codigo_disciplina","codigo_componente_curricular",
    "nome","nome_componete_curricular", "id_modulo", "modulo",
    "nome_centro", "nome_departamento", "id_curso", "id_estrutura_curricular", "percentil_media",
    "estrutura_curricular_ativo", "qtd_max_matriculas", "reprovado", "is_trancado", "risco",
    "ano_primeiro_registro", "ano_ingresso_final", "periodo_primeiro", "periodo_ingresso_final",
    "periodo_curso", "curso_ativo", "nivel_componente_curricular"
], axis=1)


# TRATANDO AS VARIAVEIS CATEGÓRICAS
df_cat_na = df_filter.select_dtypes(include=["object","str"]).columns
for na_col in df_cat_na:
    df_filter[na_col] = df_filter[na_col].fillna("NÃO INFORMADO")


# TRATANDO AS VARIAVEIS NUMERICAS
df_num_na = df_filter.select_dtypes(include=["int64","float64"]).columns
for na_col in df_num_na:
    df_filter[na_col + "_missing"] = df_filter[na_col].isna().astype(int)


df_filter['quantidade_membros_familia'] = df_filter['quantidade_membros_familia'].fillna(1)
df_filter['ano_nascimento'] = df_filter['ano_nascimento'].fillna(df_filter['ano_ingresso'] - 18)


# ACADEMIC COLS
academic_cols = [
    'ch_integralizada',
    'ch_pendente',
    'media_geral'
]
acad_mask_not_info = df_filter["curso_nome"] == "NÃO INFORMADO"

for col in academic_cols:
    # para cursos válidos -> mediana por curso
    df_filter.loc[~acad_mask_not_info, col] = df_filter.loc[~acad_mask_not_info, col].fillna(
        df_filter.loc[~acad_mask_not_info].groupby("curso_nome")[col].transform("median")
    )
    
    # para NÃO INFORMADO -> mediana global
    df_filter.loc[acad_mask_not_info, col] = df_filter.loc[acad_mask_not_info, col].fillna(
        df_filter[col].median()
    )


for na_col in df_num_na:
    df_filter[na_col] = df_filter[na_col].fillna(df_filter[na_col].median())


df_filter.info()

<class 'pandas.DataFrame'>
RangeIndex: 48447 entries, 0 to 48446
Data columns (total 69 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ano                                   48447 non-null  int64  
 1   periodo                               48447 non-null  int64  
 2   situacao                              48447 non-null  str    
 3   sexo                                  48447 non-null  str    
 4   estado_civil                          48447 non-null  str    
 5   raca_declarada                        48447 non-null  str    
 6   ano_ingresso                          48447 non-null  float64
 7   periodo_ingresso                      48447 non-null  float64
 8   status_discente                       48447 non-null  str    
 9   forma_ingresso                        48447 non-null  str    
 10  quantidade_membros_familia            48447 non-null  float64
 11  ch_integralizada          

In [94]:
df_filter.to_parquet('./files/results/data_finished.parquet', engine='pyarrow')